# Lesson 6a: Manifold Learning — Theory

## Overview
Manifold learning finds low-dimensional representations of high-dimensional data by preserving local and global structure. This lesson covers the manifold hypothesis, t-SNE and UMAP algorithms, and their mathematical foundations.

**Learning Goals:**
- Understand the manifold hypothesis
- Derive t-SNE (t-Distributed Stochastic Neighbor Embedding)
- Understand UMAP (Uniform Manifold Approximation and Projection)
- Compare manifold learning to linear methods (PCA)
- Recognize visualization artifacts

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.special import logsumexp
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Part 1: The Manifold Hypothesis

**Manifold hypothesis**: High-dimensional data lies on a low-dimensional manifold embedded in high-dimensional space.

### Example: Swiss Roll
A 2D manifold (rectangle) rolled up in 3D space. Euclidean distance misleads; geodesic distance along the surface is what matters.

In [ ]:
# Generate Swiss roll
n = 500
t = np.random.uniform(0, 4*np.pi, n)
height = np.random.uniform(-1, 1, n)
noise = np.random.randn(n, 3) * 0.1

X_swiss = np.column_stack([
    t * np.cos(t),
    height,
    t * np.sin(t)
])
X_swiss += noise

# Plot
fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(131, projection='3d')
ax.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], c=t, cmap='hsv', alpha=0.6)
ax.set_title('Swiss Roll in 3D')

# PCA (linear) projection
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_swiss)
ax = fig.add_subplot(132)
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=t, cmap='hsv', alpha=0.6)
ax.set_title('PCA (Linear) — Fails')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')

# Ideal (true 2D coordinates)
ax = fig.add_subplot(133)
ax.scatter(t, height, c=t, cmap='hsv', alpha=0.6)
ax.set_title('True 2D Manifold')
ax.set_xlabel('t')
ax.set_ylabel('height')

plt.tight_layout()
plt.show()

print("PCA fails: projects to a 2D ball")
print("Manifold learning should unfold the structure")

## Part 2: t-SNE (t-Distributed Stochastic Neighbor Embedding)

### Key Insight
Convert high-dimensional distances to probabilities (local structure), then optimize a low-dimensional representation to match those probabilities.

### Three Steps:
1. **SNE**: Define pairwise similarities in high-D and low-D spaces
2. **Symmetric SNE**: Make similarities symmetric
3. **t-SNE**: Use Student-t kernel (instead of Gaussian) for heavier tails

In [ ]:
# Simplified t-SNE math
# High-D similarities: p_ij = exp(-||x_i - x_j||^2 / 2σ²) / Z
# Low-D similarities: q_ij = (1 + ||y_i - y_j||²)^-1 / Z  (Student-t kernel)
# Loss: KL divergence between P and Q distributions

# Example: Simple dataset
n = 50
X = np.random.randn(n, 10)

# Compute pairwise distances
distances = pdist(X, metric='euclidean')
D = squareform(distances)

# Convert to similarities (softmax)
beta = 1.0  # Inverse temperature
P = np.exp(-beta * D**2)
P = P / P.sum(axis=1, keepdims=True)  # Normalize rows

# Make symmetric
P_sym = (P + P.T) / 2

print(f"High-D similarity matrix P shape: {P_sym.shape}")
print(f"P is a probability distribution: {P_sym[0].sum():.4f} (should be ~2.0 for symmetry)")
print(f"\nPerplexity intuition: roughly the effective number of neighbors")
print(f"Typical: 5-50 neighbors")

## Part 3: UMAP (Uniform Manifold Approximation and Projection)

### Key Differences from t-SNE
- Uses fuzzy topological structures (rather than probability distributions)
- Preserves both local AND global structure (t-SNE focuses on local)
- Faster (uses approximate nearest neighbor)
- More theoretically grounded (differential geometry)

### Core Idea
Model high-D and low-D data as fuzzy graphs, then optimize the low-D graph to match the high-D graph.

In [ ]:
# UMAP vs t-SNE intuition
print("t-SNE:")
print("  - Converts distances to probabilities")
print("  - Optimizes KL divergence")
print("  - Strong local structure preservation")
print("  - Weaker global structure (known issue)")
print()
print("UMAP:")
print("  - Fuzzy topological representation")
print("  - Graph-based optimization")
print("  - Preserves both local AND global structure")
print("  - Faster: ~30-200x speedup vs t-SNE")
print("  - More stable: results less sensitive to parameters")

## Part 4: Caveats — Visualization Artifacts

### Critical Warning
**t-SNE and UMAP optimizations can create spurious clusters.** Features you see in a 2D plot may not reflect true structure.

### Common Artifacts
1. **Distance metric sensitivity**: Euclidean distance may not match your data's geometry
2. **Crowding problem**: Local structures get compressed at edges
3. **Hyperparameter artifacts**: Different perplexity/n_neighbors yield different structures
4. **Clustering illusion**: Random data can appear clustered

In [ ]:
# Demonstrate clustering illusion with random data
np.random.seed(42)
n = 500

# Generate RANDOM data (no true structure)
X_random = np.random.randn(n, 50)

# Apply dimensionality reduction techniques
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_random)

# t-SNE (warning: this can be slow)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_random)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5)
axes[0].set_title('PCA of Random Data (no structure)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.5)
axes[1].set_title('t-SNE of Random Data (FAKE clusters!)')
axes[1].set_xlabel('tSNE1')
axes[1].set_ylabel('tSNE2')

plt.tight_layout()
plt.show()

print("⚠️  t-SNE creates FAKE cluster structure in random data!")
print("✓ PCA shows true structure (none).")
print("\n=> Always verify with domain knowledge, not just visualization.")

## Part 5: Comparison Summary

| Aspect | PCA | t-SNE | UMAP |
|--------|-----|-------|------|
| **Type** | Linear | Nonlinear | Nonlinear |
| **Local structure** | No | Excellent | Excellent |
| **Global structure** | Excellent | Weak | Excellent |
| **Speed** | Fast | Slow (~O(n²)) | Fast (~O(n log n)) |
| **Stability** | High | Low (hyperparameter sensitive) | High |
| **Artifacts** | None | Common (spurious clusters) | Fewer (but possible) |
| **Use case** | Preprocessing, dimensionality | Exploration (with caveats) | Exploration (recommended) |

### When to Use Each
- **PCA**: Preprocessing, downstream tasks, when you trust Euclidean distance
- **t-SNE**: Quick exploration (knowing clusters may be fake)
- **UMAP**: Exploration with better global structure, faster on large data

## Summary

### Key Concepts
1. **Manifold hypothesis**: High-D data lies on low-D manifold
2. **t-SNE**: Converts distances to probabilities, optimizes local structure
3. **UMAP**: Fuzzy graph approach, preserves local + global structure
4. **Critical caveat**: Visualization artifacts are REAL and common
5. **Best practice**: Verify manifold learning results with domain knowledge

### Next Steps
- TASK-UL14 (Practical): scikit-learn t-SNE and UMAP on real datasets
- TASK-UL15: Anomaly Detection
- Advanced: Parametric t-SNE, UMAP uncertainty